**SCAN dataset**

[data available here](https://github.com/brendenlake/SCAN)

---



In [10]:
import os
import re
import math
import requests

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Using device: cuda


***downloading SCAN from GitHub***

In [11]:
def download_github_folder(repo_owner, repo_name, folder_path, output_dir):
    """
    Download a folder from a public GitHub repo using the GitHub API.
    Saves all files (recursively) into output_dir.
    """
    api_url = f"https://api.github.com/repos/{repo_owner}/{repo_name}/contents/{folder_path}"
    response = requests.get(api_url)
    response.raise_for_status()
    items = response.json()

    os.makedirs(output_dir, exist_ok=True)

    for item in items:
        if item["type"] == "file":
            file_url = item["download_url"]
            file_data = requests.get(file_url).content
            out_path = os.path.join(output_dir, item["name"])

            with open(out_path, "wb") as f:
                f.write(file_data)
            print(f"Downloaded file: {out_path}")

        elif item["type"] == "dir":
            # Recursively download subfolders
            download_github_folder(
                repo_owner,
                repo_name,
                item["path"],
                os.path.join(output_dir, item["name"])
            )

def download_scan_simple_split():
    """
    Convenience wrapper that downloads SCAN simple_split
    into data/simple_split.
    """
    output_dir = os.path.join("data", "simple_split")

    # Skip if already downloaded
    train_path = os.path.join(output_dir, "tasks_train_simple.txt")
    test_path  = os.path.join(output_dir, "tasks_test_simple.txt")

    if os.path.exists(train_path) and os.path.exists(test_path):
        print("SCAN simple_split already downloaded.")
        return output_dir

    print("Downloading SCAN simple_split from GitHub...")
    download_github_folder(
        repo_owner="brendenlake",
        repo_name="SCAN",
        folder_path="simple_split",
        output_dir=output_dir
    )
    return output_dir

***loading SCAN data***

In [12]:
def load_scan_split(path):
    """
    Load SCAN tasks_*_simple.txt file.
    Each line: 'IN: ... OUT: ...'
    Returns: list of tokenized inputs, list of tokenized outputs.
    """
    inputs = []
    outputs = []

    with open(path, "r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            # Extract command after 'IN: ' and before ' OUT:'
            inp = re.findall(r"IN: (.*) OUT:", line)[0]
            out = line.split("OUT: ")[1]

            inputs.append(inp.split())
            outputs.append(out.split())

    return inputs, outputs


scan_dir = download_scan_simple_split()

train_path = os.path.join(scan_dir, "tasks_train_simple.txt")
test_path  = os.path.join(scan_dir, "tasks_test_simple.txt")

train_inp, train_out = load_scan_split(train_path)
test_inp,  test_out  = load_scan_split(test_path)

print(f"Train samples: {len(train_inp)}")
print(f"Test samples:  {len(test_inp)}")

SCAN simple_split already downloaded.
Train samples: 16728
Test samples:  4182


***vocab builder***

In [13]:
PAD = 0
BOS = 1
EOS = 2

def build_vocab(seqs):
    """
    Build mapping token -> id, including PAD/BOS/EOS.
    """
    vocab = {"<PAD>": PAD, "<BOS>": BOS, "<EOS>": EOS}
    idx = 3
    for seq in seqs:
        for tok in seq:
            if tok not in vocab:
                vocab[tok] = idx
                idx += 1
    return vocab

src_vocab = build_vocab(train_inp)
tgt_vocab = build_vocab(train_out)

src_ivocab = {v: k for k, v in src_vocab.items()}
tgt_ivocab = {v: k for k, v in tgt_vocab.items()}

print("Source vocab size:", len(src_vocab))
print("Target vocab size:", len(tgt_vocab))

Source vocab size: 16
Target vocab size: 9


***dataset & dataloader***

In [14]:
def encode(seq, vocab):
    return [vocab[t] for t in seq]

def make_decoder_inputs(seq_ids):
    """
    seq_ids: [y1, y2, ..., yN]
    return:
      tgt_in  = [BOS, y1, ..., yN]
      tgt_out = [y1, ..., yN, EOS]
    """
    return [BOS] + seq_ids, seq_ids + [EOS]


class ScanDataset(Dataset):
    """
    Each item:
      src:     encoded source command
      tgt_in:  decoder input sequence (with BOS)
      tgt_out: decoder label sequence (with EOS)
    """
    def __init__(self, inputs, outputs, src_vocab, tgt_vocab):
        self.inputs = inputs
        self.outputs = outputs
        self.src_vocab = src_vocab
        self.tgt_vocab = tgt_vocab

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        src_ids = encode(self.inputs[idx], self.src_vocab)
        tgt_ids = encode(self.outputs[idx], self.tgt_vocab)
        tgt_in, tgt_out = make_decoder_inputs(tgt_ids)
        return (
            torch.tensor(src_ids, dtype=torch.long),
            torch.tensor(tgt_in, dtype=torch.long),
            torch.tensor(tgt_out, dtype=torch.long),
        )


def collate_fn(batch):
    """
    Pads variable-length src/tgt sequences into tensors of shape:
      src:     [B, max_src_len]
      tgt_in:  [B, max_tgt_len]
      tgt_out: [B, max_tgt_len]
    """
    srcs, tgt_ins, tgt_outs = zip(*batch)
    B = len(batch)

    max_src = max(len(s) for s in srcs)
    max_tgt = max(len(t) for t in tgt_ins)

    src_batch = torch.full((B, max_src), PAD, dtype=torch.long)
    tgt_in_batch = torch.full((B, max_tgt), PAD, dtype=torch.long)
    tgt_out_batch = torch.full((B, max_tgt), PAD, dtype=torch.long)

    for i in range(B):
        src_batch[i, :len(srcs[i])] = srcs[i]
        tgt_in_batch[i, :len(tgt_ins[i])] = tgt_ins[i]
        tgt_out_batch[i, :len(tgt_outs[i])] = tgt_outs[i]

    return src_batch, tgt_in_batch, tgt_out_batch


train_ds = ScanDataset(train_inp, train_out, src_vocab, tgt_vocab)
test_ds  = ScanDataset(test_inp,  test_out,  src_vocab, tgt_vocab)

train_dl = DataLoader(train_ds, batch_size=64, shuffle=True,  collate_fn=collate_fn)
test_dl  = DataLoader(test_ds,  batch_size=64, shuffle=False, collate_fn=collate_fn)

# Quick sanity check
src_b, tgt_in_b, tgt_out_b = next(iter(train_dl))
print("Batch shapes:", src_b.shape, tgt_in_b.shape, tgt_out_b.shape)

Batch shapes: torch.Size([64, 9]) torch.Size([64, 34]) torch.Size([64, 34])


# TRANSFORMER

In [16]:
class MultiHeadAttention(nn.Module):
    def __init__(self, emb_dim, num_heads):
        super().__init__()
        # TODO
        assert emb_dim % num_heads == 0, "emb_dim must be divisible by num_heads"

        # Might look redundant, because: self.num_heads * self.head_dim == emb_dim
        
        self.emb_dim = emb_dim
        self.num_heads = num_heads
        self.head_dim = emb_dim // num_heads

        self.linear_value = nn.Linear(self.emb_dim, self.num_heads * self.head_dim)
        self.linear_key = nn.Linear(self.emb_dim, self.num_heads * self.head_dim)
        self.linear_query = nn.Linear(self.emb_dim, self.num_heads * self.head_dim)
        
        self.linear_output = nn.Linear(self.num_heads * self.head_dim, self.emb_dim)

    def forward(self, query, key, value, mask=None):
        batch_size = query.size(0)
        q_len = query.size(1)
        k_len = key.size(1)
        v_len = value.size(1)
        emb_dim = query.size(2)

        assert emb_dim == self.emb_dim, (
            f"Expected emb_dim={self.emb_dim}, but got {emb_dim}"
        )

        # Reminder: PyTorch tensors are backed by
        # 1 flat, contiguous array in memory
        # + shape metadata
        # + stride metadat
        # `view` creates a new interpretation of the same memory (no data copied),
        # giving it a different logical structure.
    
        # 1. Linear projections
        Q = self.linear_query(query)
        K = self.linear_key(key)
        V = self.linear_value(value)

        # 2. Split embedding dimension into multiple heads
        # Change the "view" of the data (no computation, only reshaping):
        # [batch, seq_len, emb_dim]
        # -> split emb_dim into (num_heads × head_dim)
        # -> [batch, seq_len, num_heads, head_dim]
        # This is valid because emb_dim == num_heads * head_dim
        # Example: [2, 4, 8] with 2 heads -> [2, 4, 2, 4]
        Q = Q.view(batch_size, q_len, self.num_heads, self.head_dim)
        K = K.view(batch_size, k_len, self.num_heads, self.head_dim)
        V = V.view(batch_size, v_len, self.num_heads, self.head_dim)

        # Rearrange tensor dimensions so that attention is computed per head:
        # Move num_heads in front of seq_len, because torch.matmul operates
        # on the last two dimensions and treats all earlier ones as batch dimensions.
        # [batch, seq_len, num_heads, head_dim]
        # -> [batch, num_heads, seq_len, head_dim]
        Q = Q.permute(0, 2, 1, 3)
        K = K.permute(0, 2, 1, 3)
        V = V.permute(0, 2, 1, 3)

        # 3. Scaled dot-product attention
        # Transpose keys so that matrix multiplication works:
        # [batch, heads, seq_len, head_dim]
        # -> [batch, heads, head_dim, seq_len]
        # This allows each query vector to dot-product with all key vectors
        K_t = K.transpose(-2, -1)

        key_out = torch.matmul(Q, K_t)
        key_out = key_out / math.sqrt(self.head_dim)

        # 4. Mask (optional)
        if mask is not None:
            key_out = key_out.masked_fill(mask == 0, -1e20)

        # 5. Softmax over key dimension (which tokens to attend to)
        attention = torch.softmax(key_out, dim=-1)

        # 6. Weighted sum of values
        out = torch.matmul(attention, V)

        # 7. Combine heads back into a single embedding
        # Move heads back behind sequence dimension:
        # [batch, heads, seq_len, head_dim]
        # -> [batch, seq_len, heads, head_dim]
        out = out.permute(0, 2, 1, 3)

        # Ensure memory is contiguous before reshaping
        out = out.contiguous()

        # Concatenate all heads:
        # [batch, seq_len, heads, head_dim]
        # -> [batch, seq_len, emb_dim]
        # (this is where the "concat" of heads happens)
        out = out.view(batch_size, q_len, self.emb_dim)

        # 8. Final linear projection
        out = self.linear_output(out)

        return out


In [17]:
class TransformerBlock(nn.Module):
    def __init__(self, emb_dim, num_heads, dropout, forward_dim):
        super().__init__()

        self.emb_dim = emb_dim
        self.num_heads = num_heads
        self.dropout = nn.Dropout(dropout)
        self.forward_dim = forward_dim

        self.multihead_attention = MultiHeadAttention(self.emb_dim, self.num_heads)
        
        self.norm1 = nn.LayerNorm(self.emb_dim, eps=1e-6)
        self.norm2 = nn.LayerNorm(self.emb_dim, eps=1e-6)

        self.ffn = nn.Sequential(
            nn.Linear(self.emb_dim, forward_dim),
            nn.ReLU(),
            nn.Linear(forward_dim, self.emb_dim)
        )

    def forward(self, query, key, value, mask):
        # 1. Multi-head attention
        attn_out = self.multihead_attention(query, key, value, mask)

        # 2. Skip connection + normalization
        query_with_attention = (attn_out + query)
        query_with_attention = self.dropout(query_with_attention)
        query_with_attention = self.norm1(query_with_attention)
        
        # 3. Feed-forward network
        ffn_out = self.ffn(query_with_attention)

        # 4. Skip connection + normalization
        query_with_ffn_out = (ffn_out + query_with_attention)
        query_with_ffn_out = self.dropout(query_with_ffn_out)
        query_with_ffn_out = self.norm2(query_with_ffn_out)

        block_out = query_with_ffn_out

        return block_out

In [18]:
def get_sinusoid_table(max_len, emb_dim):
    def get_angle(pos, i, emb_dim):
        return pos / 10000 ** ((2 * (i // 2)) / emb_dim)

    sinusoid_table = torch.zeros(max_len, emb_dim)
    for pos in range(max_len):
        for i in range(emb_dim):
            if i % 2 == 0:
                sinusoid_table[pos, i] = math.sin(get_angle(pos, i, emb_dim))
            else:
                sinusoid_table[pos, i] = math.cos(get_angle(pos, i, emb_dim))
    return sinusoid_table

In [19]:
class Encoder(nn.Module):
    def __init__(
        self,
        vocab_size,
        emb_dim,
        num_layers,
        num_heads,
        forward_dim,
        dropout,
        max_len,
    ):
        super().__init__()

        
        # Token embeddings
        self.token_embedding = nn.Embedding(vocab_size, emb_dim)

        # Positional encodings (sinusoid, frozen)
        pos_table = get_sinusoid_table(max_len + 1, emb_dim)
        self.position_embedding = nn.Embedding.from_pretrained(
            pos_table,
            freeze=True
        )

        # Make a dropout layer
        self.dropout = nn.Dropout(dropout)
        
        # Transformer blocks
        self.layers = nn.ModuleList(
            [
                TransformerBlock(
                    emb_dim=emb_dim,
                    num_heads=num_heads,
                    dropout=dropout,
                    forward_dim=forward_dim
                )
                for _ in range(num_layers)
            ]
        )

    def forward(self, x, mask):
        batch_size, seq_len = x.shape

        # Create position indices [1..seq_len] for each batch element
        positions = torch.arange(seq_len)                # [seq_len]
        positions = positions.unsqueeze(0)               # add batch dim -> [1, seq_len]
        positions = positions.expand(batch_size, seq_len)  # repeat for batch -> [batch, seq_len]
        positions = positions + 1                         # reserve 0 for [PAD]
        positions = positions.to(x.device)                # move to same device (cpu/gpu) as input

        # Embeddings
        token_emb = self.token_embedding(x)
        pos_emb = self.position_embedding(positions)

        # Sum + dropout
        token_pos_emb = token_emb + pos_emb
        token_pos_emb = self.dropout(token_pos_emb)

        encoder_out = token_pos_emb

        # Transformer blocks
        for layer in self.layers:
            encoder_out = layer(encoder_out, encoder_out, encoder_out, mask)

        return encoder_out

In [20]:
class DecoderBlock(nn.Module):
    def __init__(self, emb_dim, num_heads, forward_dim, dropout):
        super().__init__()
        
        # Masked self-attention (decoder attends to itself)
        self.multihead_attention = MultiHeadAttention(emb_dim, num_heads)

        # LayerNorm after first skip connection
        self.norm = nn.LayerNorm(emb_dim, eps=1e-6)

        # Cross-attention + FFN (reuse TransformerBlock)
        self.transformer_block = TransformerBlock(
            emb_dim=emb_dim,
            num_heads=num_heads,
            dropout=dropout,
            forward_dim=forward_dim
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, value, key, src_mask, tgt_mask):
        # 1. Masked self-attention (decoder attends to itself)
        multihead_attention_out = self.multihead_attention(x, x, x, tgt_mask)

        # 2. Skip connection + normalization
        x_with_self_attn = multihead_attention_out + x
        x_with_self_attn = self.dropout(x_with_self_attn)
        x_with_self_attn = self.norm(x_with_self_attn)

        # 3. Cross-attention + FFN (encoder-decoder attention)
        out = self.transformer_block(
            x_with_self_attn,
            key,
            value,
            src_mask
        )

        return out

In [21]:
class Decoder(nn.Module):
    def __init__(
        self,
        vocab_size,
        emb_dim,
        num_layers,
        num_heads,
        forward_dim,
        dropout,
        max_len
    ):
        super().__init__()

        self.token_embedding = nn.Embedding(vocab_size, emb_dim)
        self.position_embedding = nn.Embedding(max_len, emb_dim)

        self.dropout = nn.Dropout(dropout)

        self.layers = nn.ModuleList(
            [
                DecoderBlock(
                    emb_dim=emb_dim,
                    num_heads=num_heads,
                    forward_dim=forward_dim,
                    dropout=dropout
                )
                for _ in range(num_layers)
            ]
        )

        self.output_layer = nn.Linear(emb_dim, vocab_size)

    def forward(self, x, encoder_out, src_mask, tgt_mask):
        batch_size, seq_len = x.shape

        # Create position indices for decoder tokens
        positions = torch.arange(seq_len)                  # [seq_len]
        positions = positions.unsqueeze(0)                 # [1, seq_len]
        positions = positions.expand(batch_size, seq_len)  # [batch, seq_len]
        positions = positions.to(x.device)                 # move to same device as input  

        # Token + positional embeddings
        token_emb = self.token_embedding(x)
        pos_emb = self.position_embedding(positions)

        decoder_out = self.dropout(token_emb + pos_emb)

        # Pass through stacked Decoder blocks
        for layer in self.layers:
            decoder_out = layer(
                decoder_out,
                encoder_out,
                encoder_out,
                src_mask,
                tgt_mask
            )

        # Project to vocabulary size
        logits = self.output_layer(decoder_out)

        return logits


In [22]:
class Transformer(nn.Module):
    def __init__(
        self,
        src_vocab_size,
        tgt_vocab_size,
        src_pad_idx,
        tgt_pad_idx,
        emb_dim=512,
        num_layers=6,
        num_heads=8,
        forward_dim=2048,
        dropout=0.0,
        max_len=128,
    ):
        super().__init__()

        self.src_pad_idx = src_pad_idx
        self.tgt_pad_idx = tgt_pad_idx

        self.encoder = Encoder(
            vocab_size=src_vocab_size,
            emb_dim=emb_dim,
            num_layers=num_layers,
            num_heads=num_heads,
            forward_dim=forward_dim,
            dropout=dropout,
            max_len=max_len,
        )

        self.decoder = Decoder(
            vocab_size=tgt_vocab_size,
            emb_dim=emb_dim,
            num_layers=num_layers,
            num_heads=num_heads,
            forward_dim=forward_dim,
            dropout=dropout,
            max_len=max_len,
        )

    def create_src_mask(self, src):
        device = src.device
        # (batch_size, 1, 1, src_seq_len)
        src_mask = (src != self.src_pad_idx).unsqueeze(1).unsqueeze(2)
        return src_mask.to(device)

    def create_tgt_mask(self, tgt):
        device = tgt.device
        batch_size, tgt_len = tgt.shape
        tgt_mask = (tgt != self.tgt_pad_idx).unsqueeze(1).unsqueeze(2)
        tgt_mask = tgt_mask * torch.tril(torch.ones((tgt_len, tgt_len), device=device)).expand(
            batch_size, 1, tgt_len, tgt_len
        )
        return tgt_mask.to(device)

    def forward(self, src, tgt):
        # Create masks
        src_mask = self.create_src_mask(src)
        tgt_mask = self.create_tgt_mask(tgt)

        # Encode source sequence
        encoder_out = self.encoder(src, src_mask)

        # Decode target sequence
        out = self.decoder(
            tgt,
            encoder_out,
            src_mask,
            tgt_mask
        )

        return out

# TRAINING SETUP & EVALUATION

In [27]:
########################################
# Evaluation utilities (SCAN)
########################################

@torch.no_grad()
def greedy_decode(model, src, max_len=100):
    model.eval()
    B = src.size(0)

    ys = torch.full(
        (B, 1),
        BOS,
        dtype=torch.long,
        device=src.device
    )

    for _ in range(max_len):
        logits = model(src, ys)
        next_token = logits[:, -1].argmax(dim=-1, keepdim=True)
        ys = torch.cat([ys, next_token], dim=1)

        if (next_token == EOS).all():
            break

    return ys


def sequence_accuracy(preds, targets):
    correct = 0
    for p, t in zip(preds, targets):
        p = p.tolist()
        t = t.tolist()

        # drop BOS from prediction if present
        if len(p) > 0 and p[0] == BOS:
            p = p[1:]

        if EOS in p: p = p[:p.index(EOS)+1]
        if EOS in t: t = t[:t.index(EOS)+1]

        if p == t:
            correct += 1
    return correct / len(targets)



@torch.no_grad()
def evaluate_sequence_accuracy(model, dataloader):
    model.eval()
    total_correct = 0
    total = 0

    for src, _, tgt_out in dataloader:
        src = src.to(device)
        tgt_out = tgt_out.to(device)

        preds = greedy_decode(model, src)

        batch_acc = sequence_accuracy(preds, tgt_out)
        total_correct += batch_acc * src.size(0)
        total += src.size(0)

    return total_correct / total


In [28]:
#training setup
model = Transformer(
    src_vocab_size=len(src_vocab),
    tgt_vocab_size=len(tgt_vocab),
    src_pad_idx=PAD,
    tgt_pad_idx=PAD,
    emb_dim=128,
    num_layers=2,
    num_heads=4,
    forward_dim=256,
    dropout=0.1,
    max_len=100,
).to(device)

criterion = torch.nn.CrossEntropyLoss(ignore_index=PAD)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

#accuracy
def tokens_accuracy(logits, targets):
    """
    logits: [B, L, V]
    targets: [B, L]
    """
    preds = logits.argmax(dim=-1)
    mask = (targets != PAD)
    correct = (preds == targets) & mask
    return correct.sum().item() / mask.sum().item()

@torch.no_grad()
def evaluate_token_level(model, dataloader):
    model.eval()
    total_loss = 0.0
    total_acc = 0.0
    steps = 0

    for src, tgt_in, tgt_out in dataloader:
        src = src.to(device)
        tgt_in = tgt_in.to(device)
        tgt_out = tgt_out.to(device)

        logits = model(src, tgt_in)

        loss = criterion(
            logits.reshape(-1, logits.size(-1)),
            tgt_out.reshape(-1)
        )

        acc = tokens_accuracy(logits, tgt_out)

        total_loss += loss.item()
        total_acc += acc
        steps += 1

    return total_loss / steps, total_acc / steps


#training loop

EPOCHS = 20

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0
    steps = 0

    for src, tgt_in, tgt_out in train_dl:
        src = src.to(device)
        tgt_in = tgt_in.to(device)
        tgt_out = tgt_out.to(device)

        optimizer.zero_grad()

        logits = model(src, tgt_in)

        loss = criterion(
            logits.reshape(-1, logits.size(-1)),
            tgt_out.reshape(-1)
        )

        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        steps += 1

    # token-level evaluation (teacher forcing)
    val_loss, val_tok_acc = evaluate_token_level(model, test_dl)

    # sequence-level evaluation (greedy decoding)
    val_seq_acc = evaluate_sequence_accuracy(model, test_dl)

    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={running_loss/steps:.4f} | "
        f"val_loss={val_loss:.4f} | "
        f"token_acc={val_tok_acc*100:.2f}% | "
        f"seq_acc={val_seq_acc*100:.2f}%"
    )


Epoch 01 | train_loss=1.1102 | val_loss=0.7239 | token_acc=72.55% | seq_acc=0.00%
Epoch 02 | train_loss=0.6800 | val_loss=0.5377 | token_acc=78.04% | seq_acc=0.31%
Epoch 03 | train_loss=0.5351 | val_loss=0.4095 | token_acc=82.56% | seq_acc=1.75%
Epoch 04 | train_loss=0.4225 | val_loss=0.2802 | token_acc=88.92% | seq_acc=9.23%
Epoch 05 | train_loss=0.2958 | val_loss=0.1923 | token_acc=92.20% | seq_acc=22.05%
Epoch 06 | train_loss=0.2196 | val_loss=0.1459 | token_acc=93.89% | seq_acc=28.24%
Epoch 07 | train_loss=0.1706 | val_loss=0.0918 | token_acc=96.07% | seq_acc=47.58%
Epoch 08 | train_loss=0.1390 | val_loss=0.0846 | token_acc=96.25% | seq_acc=47.68%
Epoch 09 | train_loss=0.1170 | val_loss=0.0854 | token_acc=96.09% | seq_acc=43.33%
Epoch 10 | train_loss=0.1012 | val_loss=0.0533 | token_acc=97.80% | seq_acc=67.53%
Epoch 11 | train_loss=0.0881 | val_loss=0.0709 | token_acc=97.02% | seq_acc=55.14%
Epoch 12 | train_loss=0.0796 | val_loss=0.0436 | token_acc=98.09% | seq_acc=71.35%
Epoch 13

In [29]:
# SANITY CHECK
src, _, tgt_out = next(iter(test_dl))
src, tgt_out = src.to(device), tgt_out.to(device)
preds = greedy_decode(model, src[:1])

print("pred ids:", preds[0].tolist())
print("gold ids:", tgt_out[0].tolist())


pred ids: [1, 3, 3, 3, 3, 3, 3, 6, 6, 6, 2]
gold ids: [3, 3, 3, 3, 3, 3, 6, 6, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


In [30]:
seq_acc = evaluate_sequence_accuracy(model, test_dl)
print(f"Sequence-level accuracy (simple split): {seq_acc*100:.2f}%")

Sequence-level accuracy (simple split): 74.77%


In [31]:
torch.save(model.state_dict(), "r_transformer_scan.pt")